In [0]:
# =============================================================
# CELDA 1: GOLD — KPIs + Resumen de Mercado Robusto
# =============================================================
import json
import boto3
from pyspark.sql import functions as F

# Credenciales: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
except Exception:
    try:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)
    except FileNotFoundError:
        raise SystemExit("❌ Credenciales no disponibles. Configura Databricks Secrets (scope='aws') o coloca aws_secrets.json.")

AWS_KEY = config["aws_access_key"]
AWS_SECRET = config["aws_secret_key"]
BUCKET = "bronce-scrap-date"

S3_OPTS = {
    "fs.s3a.access.key": AWS_KEY,
    "fs.s3a.secret.key": AWS_SECRET,
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

s3_check = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

# --- Leer Silver Deduped (output de 02b_Dedup_Cross_Portal) ---
# Fallback a master_inmuebles si master_deduped no existe aún
try:
    s3_check.list_objects_v2(
        Bucket=BUCKET, Prefix="silver/master_deduped/_delta_log/", MaxKeys=1
    )["Contents"]
    ruta_silver = f"s3a://{BUCKET}/silver/master_deduped/"
    print("📖 Leyendo Silver DEDUPED (cross-portal)")
except (KeyError, Exception):
    ruta_silver = f"s3a://{BUCKET}/silver/master_inmuebles/"
    print("📖 Leyendo Silver original (dedup no ejecutado aún)")

reader = spark.read.format("delta")
for k, v in S3_OPTS.items():
    reader = reader.option(k, v)
df_silver = reader.load(ruta_silver)

print(f"   {df_silver.count()} registros")
print(f"   Columnas: {df_silver.columns}")

# --- Leer price intelligence si existe para enriquecer el resumen ---
try:
    s3_check.list_objects_v2(
        Bucket=BUCKET, Prefix="gold/price_intelligence/_delta_log/", MaxKeys=1
    )["Contents"]
    reader_intel = spark.read.format("delta")
    for k, v in S3_OPTS.items():
        reader_intel = reader_intel.option(k, v)
    df_intel = reader_intel.load(f"s3a://{BUCKET}/gold/price_intelligence/")
    print("💰 Price intelligence disponible para enriquecer Gold")
except (KeyError, Exception):
    df_intel = None
    print("ℹ️ Price intelligence no disponible todavía")

# --- KPIs robustos por inmueble ---
df_gold_base = (
    df_silver
    .filter(
        F.col("precio_num").isNotNull() &
        (F.col("precio_num") > 0) &
        F.col("area_m2").isNotNull() &
        (F.col("area_m2") >= 20) &
        (F.col("area_m2") <= 800)
    )
    .withColumn("precio_m2", F.round(F.col("precio_num") / F.col("area_m2"), 0))
    .withColumn("city_token", F.coalesce(F.col("city_token"), F.lit("otra_ciudad")))
    .withColumn("tipo_inmueble", F.coalesce(F.col("tipo_inmueble"), F.lit("otro")))
    .withColumn(
        "zona_mercado",
        F.coalesce(
            F.when(F.length(F.col("ubicacion_norm")) > 4, F.col("ubicacion_norm")),
            F.when(F.length(F.col("ubicacion_raw")) > 4, F.lower(F.trim(F.col("ubicacion_raw")))),
            F.concat_ws(" | ", F.col("city_token"), F.col("tipo_inmueble"))
        )
    )
    .withColumn(
        "market_segment",
        F.concat_ws("__", F.col("city_token"), F.col("tipo_inmueble"))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

# Filtro robusto por precio_m2 dentro de segmento ciudad + tipo
segment_stats = (
    df_gold_base
    .groupBy("market_segment")
    .agg(
        F.count("*").alias("segment_n"),
        F.expr("percentile_approx(precio_m2, 0.05)").alias("pm2_p05_segment"),
        F.expr("percentile_approx(precio_m2, 0.95)").alias("pm2_p95_segment"),
    )
)

global_pm2 = df_gold_base.agg(
    F.expr("percentile_approx(precio_m2, 0.02)").alias("pm2_p02_global"),
    F.expr("percentile_approx(precio_m2, 0.98)").alias("pm2_p98_global"),
).first()

df_gold = (
    df_gold_base
    .join(segment_stats, "market_segment", "left")
    .withColumn(
        "pm2_lower_bound",
        F.when(F.col("segment_n") >= 15, F.col("pm2_p05_segment")).otherwise(F.lit(global_pm2["pm2_p02_global"]))
    )
    .withColumn(
        "pm2_upper_bound",
        F.when(F.col("segment_n") >= 15, F.col("pm2_p95_segment")).otherwise(F.lit(global_pm2["pm2_p98_global"]))
    )
    .filter(F.col("precio_m2").between(F.col("pm2_lower_bound"), F.col("pm2_upper_bound")))
    .drop("pm2_lower_bound", "pm2_upper_bound", "pm2_p05_segment", "pm2_p95_segment")
)

print(f"\n📊 Gold detalle robusto: {df_gold.count()} inmuebles con KPI")
display(
    df_gold.select(
        "titulo", "city_token", "zona_mercado", "tipo_inmueble",
        "precio_num", "area_m2", "precio_m2",
        "habitaciones", "banos", "garajes", "num_portales", "dispersion_pct_grupo"
    ).limit(10)
)

# --- Resumen Agregado por zona normalizada ---
df_resumen = (
    df_gold.groupBy("city_token", "zona_mercado", "tipo_inmueble")
    .agg(
        F.count("id_original").alias("total_ofertas"),
        F.expr("percentile_approx(precio_num, 0.5)").alias("precio_mediano"),
        F.expr("percentile_approx(area_m2, 0.5)").alias("area_mediana"),
        F.expr("percentile_approx(precio_m2, 0.5)").alias("valor_m2_mediano"),
        F.expr("percentile_approx(precio_m2, 0.25)").alias("valor_m2_p25"),
        F.expr("percentile_approx(precio_m2, 0.75)").alias("valor_m2_p75"),
        F.round(F.avg("habitaciones"), 1).alias("habitaciones_promedio"),
        F.round(F.avg("banos"), 1).alias("banos_promedio"),
        F.round(F.avg(F.coalesce(F.col("num_portales"), F.lit(1))), 2).alias("promedio_portales"),
        F.round(F.avg(F.coalesce(F.col("dispersion_pct_grupo"), F.lit(0.0))), 1).alias("dispersion_promedio_pct"),
        F.round(F.avg(F.when(F.coalesce(F.col("num_portales"), F.lit(1)) > 1, F.lit(1.0)).otherwise(F.lit(0.0))) * 100, 1).alias("pct_multiportal"),
    )
    .filter(F.col("total_ofertas") >= 3)
)

if df_intel is not None:
    df_oportunidades = (
        df_intel
        .withColumn("city_token", F.coalesce(F.col("city_token"), F.lit("otra_ciudad")))
        .withColumn("tipo_inmueble", F.coalesce(F.col("tipo_inmueble"), F.lit("otro")))
        .withColumn(
            "zona_mercado",
            F.coalesce(
                F.when(F.length(F.col("ubicacion_norm")) > 4, F.col("ubicacion_norm")),
                F.when(F.length(F.col("ubicacion_raw")) > 4, F.lower(F.trim(F.col("ubicacion_raw")))),
                F.concat_ws(" | ", F.col("city_token"), F.col("tipo_inmueble"))
            )
        )
        .groupBy("city_token", "zona_mercado", "tipo_inmueble")
        .agg(
            F.count("*").alias("oportunidades_cross_portal"),
            F.round(F.avg("ahorro_potencial"), 0).alias("ahorro_promedio_cross_portal"),
            F.round(F.avg("ahorro_pct"), 1).alias("ahorro_pct_promedio"),
        )
    )
    df_resumen = df_resumen.join(df_oportunidades, ["city_token", "zona_mercado", "tipo_inmueble"], "left")

df_resumen = df_resumen.orderBy(F.desc("valor_m2_mediano"), F.desc("total_ofertas"))

print(f"\n🏆 Zonas con >= 3 ofertas limpias: {df_resumen.count()}")
display(df_resumen.limit(100))

In [0]:
# =============================================================
# CELDA 2: GUARDAR — Gold Resumen (Delta) + Gold Detalle (Parquet)
# =============================================================

# Gold Resumen: agregación robusta por zona
ruta_resumen = f"s3a://{BUCKET}/gold/mercado_resumen/"
writer = df_resumen.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_resumen)
print(f"💾 Gold resumen: {ruta_resumen}")

# Gold Detalle: inmuebles limpios con KPIs robustos
ruta_app = f"s3a://{BUCKET}/gold/app_inmuebles/"
writer = df_gold.coalesce(1).write.format("parquet").mode("overwrite")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_app)
print(f"🚀 Gold detalle: {ruta_app}")